In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [ ]:
df = pd.read_csv("../data/processed/clean_data.csv")

df.head()

In [ ]:
print(df.shape)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = df[
    [
        "prix",
        "surface",
        "chambres",
        "salles_de_bain",
        "etage",
        "localisation"
    ]
]

In [ ]:
print(df.columns.tolist())

In [ ]:
df = df.dropna()

In [ ]:
df.info()

In [ ]:
y = df["prix"]
X = df[
    [
        "surface",
        "chambres",
        "salles_de_bain",
        "etage",
        "localisation"
    ]
]

In [ ]:
X = pd.get_dummies(
    X,
    columns=["localisation"],
    drop_first=True
)

In [ ]:
print(X.head())
print(y.head())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42,
    n_estimators=200
)

model.fit(X_train, y_train)

In [ ]:
df["etage"].value_counts().head(20)

In [ ]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
#Vérifier les tailles
print(X_train.shape)
print(X_test.shape)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R² :", round(r2, 4))

In [ ]:
print(df.columns.tolist())

In [ ]:
X.info()

In [ ]:
top_locations = df["localisation"].value_counts().head(30).index

df["localisation"] = df["localisation"].apply(
    lambda x: x if x in top_locations else "Autre"
)

In [ ]:
df["localisation"].nunique()

In [ ]:
print(X.shape)

# VERSION FINALE : ANALYSE DES OUTLIERS

In [ ]:
df["prix"].describe()

In [ ]:
df["surface"].describe()

In [ ]:
import matplotlib.pyplot as plt

df.boxplot(column="prix")

plt.show()

In [ ]:
df.boxplot(column="surface")

plt.show()

# Suppression des valeurs aberrantes

In [ ]:
Q1 = df["prix"].quantile(0.25)
Q3 = df["prix"].quantile(0.75)

IQR = Q3 - Q1

borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

print(borne_inf)
print(borne_sup)

In [ ]:
df_no_outliers = df[
    (df["prix"] >= borne_inf)
    &
    (df["prix"] <= borne_sup)
]

print(df.shape)
print(df_no_outliers.shape)

In [ ]:
X = df_no_outliers[
    [
        "surface",
        "chambres",
        "salles_de_bain",
        "etage"
    ]
]

y = df_no_outliers["prix"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42,
    n_estimators=200
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Calcul des métriques
mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(
    y_test,
    y_pred
)

# Affichage
print("MAE :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R² :", round(r2, 4))

# Résultat après suppression des outliers
# MAE : 299802.74
# RMSE : 406129.70
# R² : 0.3186

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Scores :", scores)
print("R² moyen :", scores.mean())
print("Écart-type :", scores.std())

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
print("Meilleurs paramètres :")
print(grid.best_params_)

print("Meilleur score CV :")
print(grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R² :", round(r2, 4))

In [ ]:
importance = pd.DataFrame({
    "Variable": X.columns,
    "Importance": best_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

In [ ]:
importance.plot(
    x="Variable",
    y="Importance",
    kind="barh",
    figsize=(8,5)
)

In [ ]:
import joblib

joblib.dump(
    best_model,
    "../models/model_prix.pkl"
)

In [ ]:
joblib.dump(
    X.columns.tolist(),
    "../models/features.pkl"
)

# Analyse des résultats

In [ ]:
importance.sort_values(
    by="Importance",
    ascending=False
)

In [ ]:
import matplotlib.pyplot as plt

importance.sort_values(
    by="Importance"
).plot(
    x="Variable",
    y="Importance",
    kind="barh",
    figsize=(10,6)
)

plt.title("Importance des variables")
plt.show()